In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import matplotlib as mpl
new_rc_params = {'text.usetex': False, "svg.fonttype": 'none'}
mpl.rcParams.update(new_rc_params)
from scipy.stats import linregress
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.inspection import PartialDependenceDisplay

In [ ]:
df=pd.read_csv('/g/korbel/olisov/Visium_Linda/stiffness/normalized/merged_11samples_50um_final.tsv', sep='\t', index_col=0)
# Number of columns to retain + Y column
n = 1000

# Calculate the number of zeros in each column
zero_counts = (df == 0).sum()

# Sort columns by the number of zeros
sorted_columns = zero_counts.sort_values().index

selected = list(sorted_columns[:n])  # top n genes

if "TFF1" not in selected:selected.append("TFF1")
if "COL12A1" not in selected:selected.append("COL12A1")

df1 = df[selected]

In [ ]:
# Split data into features and target
X = df1.drop(['mean', 'min', 'max'], axis=1)
y = df1['mean']

# Initialize the Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=0, max_depth=10, min_samples_split=10, min_samples_leaf=4)

# Define 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=0)

# Perform cross-validation and collect scores
cv_mse_scores = cross_val_score(rf_model, X, y, cv=kf, scoring='neg_mean_squared_error')
cv_r2_scores = cross_val_score(rf_model, X, y, cv=kf, scoring='r2')

# Convert MSE scores to positive values
cv_mse_scores = -cv_mse_scores

# Calculate average MSE, RMSE, and R-squared from cross-validation
avg_mse = np.mean(cv_mse_scores)
avg_rmse = math.sqrt(avg_mse)
avg_r2 = np.mean(cv_r2_scores)

print("Cross-Validation Results:")
print("Average Root Mean Squared Error:", avg_rmse)
print("Average R-squared:", avg_r2)

# Fit the model on the entire dataset
rf_model.fit(X, y)

y_pred_cv = cross_val_predict(rf_model, X, y, cv=kf)

# Feature importance
importances = rf_model.feature_importances_
features = X.columns

# Create a DataFrame for importances and feature names
importance_df = pd.DataFrame({'Feature': features, 'Importance': importances})

# Sort the features by importance
important_features = importance_df.sort_values(by='Importance', ascending=False)

In [ ]:
# -------------------------------
# 3. Scatterplot: Predicted vs Observed stiffness (log10)
# -------------------------------
# Set fixed figure dimensions
fig_width_mm = 43.5
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch * 0.8

# Apply log10 transform
y_log = np.log10(y)
y_pred_log = np.log10(y_pred_cv)

# Create figure and axis
fig, ax = plt.subplots(figsize=(fig_width_inch, fig_width_inch), dpi=600)
sns.scatterplot(x=y_pred_log, y=y_log, color='grey', s=3, ax=ax)

# Correlation line (in linear space since mean is already log-transformed)
slope, intercept, r_value, p_value, std_err = linregress(y_pred_log, y_log)
x_sorted = np.linspace(y_pred_log.min(), y_pred_log.max(), 500)
y_sorted = intercept + slope * x_sorted
ax.plot(x_sorted, y_sorted, color='red', linewidth=0.5)

# Annotate r and p value
ax.text(0.5, 0.95, f'$r$ = {r_value:.2f}', transform=ax.transAxes,
        ha='center', va='bottom', fontsize=3)
ax.text(0.5, 0.90, f'$p$ = {p_value:.2e}', transform=ax.transAxes,
        ha='center', va='bottom', fontsize=3)

plt.xlabel("log₁₀Predicted", fontsize=3)
plt.ylabel("log₁₀Observed", fontsize=3)
plt.title("Random Forest stiffness prediction", fontsize=3)
ax.set_xticks((2.2,2.7,3.2))
ax.set_yticks((1.5,2,2.5,3,3.5))
ax.set_xlim(left=2.1)
ax.tick_params(axis='both', which='major', pad=0.5, width=0.3, labelsize=3)

sns.despine(top=True, right=True)
plt.tight_layout()
plt.savefig(f'/g/korbel/olisov/Visium_Linda/paper_figures/11samples/RF_correlation_11.svg', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
important_features=important_features[important_features['Importance']>0]

In [ ]:
important_features

In [ ]:
important_features.to_csv('/g/korbel/olisov/Visium_Linda/stiffness/regression/final_RF_11samples_topgenes_um50.tsv', sep='\t', index=False)

In [ ]:
# Select top 15 features for readability
fig_width_mm = 43.5
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch * 0.8
plt.figure(figsize=(fig_width_inch, fig_height_inch), dpi=600)

top_n = 5
top_features = important_features.head(top_n)

sns.barplot(data=top_features, x='Importance', y='Feature', palette='viridis_r')
plt.tick_params(axis='both', which='major', pad=0.5, width=0.3, labelsize=3)
plt.xlabel('Feature Importance', fontsize=3)
plt.ylabel('')
plt.title(f'Random Forest Feature Importances', fontsize=3)
sns.despine(top=True, right=True)
plt.tight_layout()
plt.savefig(f'/g/korbel/olisov/Visium_Linda/paper_figures/11samples/RF_importance.svg', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
#partial dependence
feature='TGFBI'

fig_width_mm = 43.5
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch
plt.figure(figsize=(fig_width_inch, fig_height_inch), dpi=600)

# Plot partial dependence using PartialDependenceDisplay
pd_display = PartialDependenceDisplay.from_estimator(rf_model, X, features=[feature], grid_resolution=100)
#pd_display.axes_[0, 0].collections[0].remove()

# Overlay scatter plot of the actual feature values
#plt.scatter(X[feature], y, alpha=0.7, color='red', s=10)
plt.xlabel(f"{feature} expression")
plt.ylabel("Predicted stiffness")
sns.despine(top=True, right=True)
plt.ylim(400, 700)
plt.xlim(-0.1, 4.5)

# Show the plot
plt.savefig(f'/g/korbel/olisov/Visium_Linda/paper_figures/11samples/partial_dependence_{feature}.svg', bbox_inches='tight')
plt.show()

In [ ]:
feature='TFF3'

fig_width_mm = 43.5
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch
plt.figure(figsize=(fig_width_inch, fig_height_inch), dpi=600)

# Plot partial dependence using PartialDependenceDisplay
pd_display = PartialDependenceDisplay.from_estimator(rf_model, X, features=[feature], grid_resolution=100)
#pd_display.axes_[0, 0].collections[0].remove()

# Overlay scatter plot of the actual feature values
#plt.scatter(X[feature], y, alpha=0.7, color='red', s=10)
plt.xlabel(f"{feature} expression")
plt.ylabel("Predicted stiffness")
sns.despine(top=True, right=True)
plt.ylim(380, 900)

# Show the plot
plt.savefig(f'/g/korbel/olisov/Visium_Linda/paper_figures/11samples/partial_dependence_{feature}.svg', bbox_inches='tight')
plt.show()

In [ ]:
feature='TFF1'

fig_width_mm = 43.5
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch
plt.figure(figsize=(fig_width_inch, fig_height_inch), dpi=600)

# Plot partial dependence using PartialDependenceDisplay
pd_display = PartialDependenceDisplay.from_estimator(rf_model, X, features=[feature], grid_resolution=100)
#pd_display.axes_[0, 0].collections[0].remove()

# Overlay scatter plot of the actual feature values
#plt.scatter(X[feature], y, alpha=0.7, color='red', s=10)
plt.xlabel(f"{feature} expression")
plt.ylabel("Predicted stiffness")
sns.despine(top=True, right=True)
plt.ylim(380, 950)
plt.xlim(-0.2, 4.5)

# Show the plot
plt.savefig(f'/g/korbel/olisov/Visium_Linda/paper_figures/11samples/partial_dependence_{feature}.svg', bbox_inches='tight')
plt.show()

In [ ]:
feature='PRAP1'

fig_width_mm = 43.5
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch
plt.figure(figsize=(fig_width_inch, fig_height_inch), dpi=600)

# Plot partial dependence using PartialDependenceDisplay
pd_display = PartialDependenceDisplay.from_estimator(rf_model, X, features=[feature], grid_resolution=100)
#pd_display.axes_[0, 0].collections[0].remove()

# Overlay scatter plot of the actual feature values
#plt.scatter(X[feature], y, alpha=0.7, color='red', s=10)
plt.xlabel(f"{feature} expression")
plt.ylabel("Predicted stiffness")
sns.despine(top=True, right=True)
plt.ylim(400, 660)

# Show the plot
plt.savefig(f'/g/korbel/olisov/Visium_Linda/paper_figures/11samples/partial_dependence_{feature}.svg', bbox_inches='tight')
plt.show()

In [ ]:
feature='GAPDH'

fig_width_mm = 43.5
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch
plt.figure(figsize=(fig_width_inch, fig_height_inch), dpi=600)

# Plot partial dependence using PartialDependenceDisplay
pd_display = PartialDependenceDisplay.from_estimator(rf_model, X, features=[feature], grid_resolution=100)
#pd_display.axes_[0, 0].collections[0].remove()

# Overlay scatter plot of the actual feature values
#plt.scatter(X[feature], y, alpha=0.7, color='red', s=10)
plt.xlabel(f"{feature} expression")
plt.ylabel("Predicted stiffness")
sns.despine(top=True, right=True)
plt.ylim(400, 760)

# Show the plot
plt.savefig(f'/g/korbel/olisov/Visium_Linda/paper_figures/11samples/partial_dependence_{feature}.svg', bbox_inches='tight')
plt.show()

In [ ]:
feature='COL12A1'

fig_width_mm = 43.5
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch
plt.figure(figsize=(fig_width_inch, fig_height_inch), dpi=600)

# Plot partial dependence using PartialDependenceDisplay
pd_display = PartialDependenceDisplay.from_estimator(rf_model, X, features=[feature], grid_resolution=100)
#pd_display.axes_[0, 0].collections[0].remove()

# Overlay scatter plot of the actual feature values
#plt.scatter(X[feature], y, alpha=0.7, color='red', s=10)
plt.xlabel(f"{feature} expression")
plt.ylabel("Predicted stiffness")
sns.despine(top=True, right=True)
plt.ylim(400, 660)

# Show the plot
plt.savefig(f'/g/korbel/olisov/Visium_Linda/paper_figures/11samples/partial_dependence_{feature}.svg', bbox_inches='tight')
plt.show()